# Eval100 test run: chart data as DataFrames

Loads the data behind every chart in `data/results/2026-11-18_claude_test_eval100_1/analysis/charts/`.
Each chart's DataFrame holds the same rows and columns that `Optimizer/charts.py` plots. Run from `notebooks/`.

In [1]:
import json
from pathlib import Path
import pandas as pd

RUN = Path("../data/results/2026-11-18_claude_test_eval100_1")
ANALYSIS = RUN / "analysis"

meta = json.loads((RUN / "run_meta.json").read_text())
ARMS, LAST = list(meta["arms"]), int(meta["retries"])
FB = [a for a in ARMS if a != "control"]

t = {p.stem: pd.read_csv(p) for p in sorted(ANALYSIS.glob("*.csv"))}
{name: df.shape for name, df in t.items()}

{'arm_outcomes': (4, 33),
 'attempts': (1700, 29),
 'by_group': (323, 16),
 'cells': (1, 13),
 'curves': (48, 15),
 'failure_moves': (17, 4),
 'failure_types': (17, 7),
 'main': (17, 22),
 'paired_tests': (42, 13),
 'retry_value': (12, 12),
 'stops': (4, 6),
 'tasks': (100, 31),
 'transitions': (12, 13)}

## 01 Accuracy by attempt (`curves.csv`, Panacea line from `main.csv`)

In [2]:
c, m = t["curves"], t["main"]
df_01 = c[c["scope"].isin(["all", "paired"])][["scope", "arm", "attempt", "n", "hidden_pass@1", "ci_low", "ci_high"]]
df_01_panacea = m[(m["system"] == "panacea") & (m["attempt"] == 0)][["system", "hidden_pass@1"]]
df_01

,scope,arm,attempt,n,hidden_pass@1,ci_low,ci_high
0,all,control,0,100,0.770,0.678,0.842
1,all,control,1,100,0.790,0.700,0.858
2,all,control,2,100,0.810,0.722,0.875
3,all,control,3,100,0.810,0.722,0.875
4,all,bare_retry,0,100,0.770,0.678,0.842
5,all,bare_retry,1,100,0.800,0.711,0.867
6,all,bare_retry,2,100,0.810,0.722,0.875
7,all,bare_retry,3,100,0.830,0.745,0.891
8,all,reflect_fix,0,100,0.770,0.678,0.842
9,all,reflect_fix,1,100,0.830,0.745,0.891


In [13]:
df_01[df_01['n']==18]

,scope,arm,attempt,n,hidden_pass@1,ci_low,ci_high
16,paired,control,0,18,0.000,0.000,0.176
17,paired,control,1,18,0.111,0.031,0.328
18,paired,control,2,18,0.222,0.090,0.452
19,paired,control,3,18,0.222,0.090,0.452
20,paired,bare_retry,0,18,0.000,0.000,0.176
21,paired,bare_retry,1,18,0.167,0.058,0.392
22,paired,bare_retry,2,18,0.222,0.090,0.452
23,paired,bare_retry,3,18,0.333,0.163,0.563
24,paired,reflect_fix,0,18,0.000,0.000,0.176
25,paired,reflect_fix,1,18,0.333,0.163,0.563


## 02 Gain vs control (`curves.csv`)

In [3]:
df_02 = c[c["scope"].isin(["all", "paired"]) & (c["attempt"] > 0) & c["arm"].isin(FB)][
    ["scope", "arm", "attempt", "n", "gain_vs_control", "gain_ci_low", "gain_ci_high"]]
df_02

,scope,arm,attempt,n,gain_vs_control,gain_ci_low,gain_ci_high
5,all,bare_retry,1,100,0.010,0.000,0.030
6,all,bare_retry,2,100,0.000,-0.030,0.030
7,all,bare_retry,3,100,0.020,-0.010,0.060
9,all,reflect_fix,1,100,0.040,0.000,0.090
10,all,reflect_fix,2,100,0.040,0.000,0.090
11,all,reflect_fix,3,100,0.040,0.000,0.090
13,all,critic,1,100,0.040,0.000,0.090
14,all,critic,2,100,0.030,-0.010,0.070
15,all,critic,3,100,0.030,-0.010,0.070
21,paired,bare_retry,1,18,0.056,0.000,0.167


## 03 Rubric levels (`main.csv`)

In [14]:
rows = [("zero_shot", 0)] + [(a, k) for a in ARMS for k in range(1, LAST + 1)]
if "panacea" in set(m["system"]):
    rows.append(("panacea", 0))
level_cols = [f"level_{lv}" for lv in range(5)] + ["level_none"]
df_03 = pd.concat([m[(m["system"] == s) & (m["attempt"] == k)] for s, k in rows])[["system", "attempt", "n"] + level_cols]
df_03

,system,attempt,n,level_0,level_1,level_2,level_3,level_4,level_none
0,zero_shot,0,100,0,12,3,8,77,0
2,control,1,100,1,6,6,8,79,0
3,control,2,100,1,7,3,8,81,0
4,control,3,100,1,6,2,10,81,0
6,bare_retry,1,100,1,6,7,6,80,0
7,bare_retry,2,100,1,6,6,6,81,0
8,bare_retry,3,100,1,7,4,5,83,0
10,reflect_fix,1,100,0,3,6,8,83,0
11,reflect_fix,2,100,1,2,4,8,85,0
12,reflect_fix,3,100,0,4,4,7,85,0


## 04–06 By group (`by_group.csv`, C + D tasks)

In [15]:
bg = t["by_group"]
def by_group(grouping, scope="paired"):
    d = bg[(bg["grouping"] == grouping) & (bg["scope"] == scope) & bg["system"].isin(ARMS)]
    return d.rename(columns={"system": "arm"})[["group", "arm", "attempt", "n", "hidden_pass@1", "gain_vs_control"]]

df_04 = by_group("source_difficulty")
df_05 = by_group("attempt0_failure_type")
df_06 = by_group("test_format")
df_04

,group,arm,attempt,n,hidden_pass@1,gain_vs_control
120,livecodebench/hard,control,0,9,0.000,0.0
121,livecodebench/hard,control,1,9,0.000,0.0
122,livecodebench/hard,control,2,9,0.111,0.0
123,livecodebench/hard,control,3,9,0.111,0.0
124,livecodebench/hard,bare_retry,0,9,0.000,0.0
...,...,...,...,...,...,...
182,mbpp/medium,reflect_fix,3,2,1.000,1.0
183,mbpp/medium,critic,0,2,0.000,0.0
184,mbpp/medium,critic,1,2,1.000,1.0
185,mbpp/medium,critic,2,2,1.000,1.0


In [16]:
df_05

,group,arm,attempt,n,hidden_pass@1,gain_vs_control
1,runtime_error,control,0,1,0.000,0.000
2,runtime_error,control,1,1,0.000,0.000
3,runtime_error,control,2,1,0.000,0.000
4,runtime_error,control,3,1,0.000,0.000
5,runtime_error,bare_retry,0,1,0.000,0.000
6,runtime_error,bare_retry,1,1,0.000,0.000
7,runtime_error,bare_retry,2,1,0.000,0.000
8,runtime_error,bare_retry,3,1,0.000,0.000
9,runtime_error,reflect_fix,0,1,0.000,0.000
10,runtime_error,reflect_fix,1,1,0.000,0.000


In [17]:
df_06

,group,arm,attempt,n,hidden_pass@1,gain_vs_control
273,assert_list,control,0,6,0.000,0.000
274,assert_list,control,1,6,0.167,0.000
275,assert_list,control,2,6,0.167,0.000
276,assert_list,control,3,6,0.167,0.000
277,assert_list,bare_retry,0,6,0.000,0.000
278,assert_list,bare_retry,1,6,0.333,0.166
279,assert_list,bare_retry,2,6,0.333,0.166
280,assert_list,bare_retry,3,6,0.500,0.333
281,assert_list,reflect_fix,0,6,0.000,0.000
282,assert_list,reflect_fix,1,6,0.667,0.500


## 08 Repairs and breaks (`arm_outcomes.csv`)

In [18]:
o = t["arm_outcomes"]
df_08 = o[["arm", "n_D", "n_C", "repair_rate", "false_fix_share", "break_rate", "regression_rate_per_retry"]]
df_08

,arm,n_D,n_C,repair_rate,false_fix_share,break_rate,regression_rate_per_retry
0,control,18,0,0.222,0.200,NaN,0.083
1,bare_retry,18,0,0.333,0.000,NaN,0.128
2,reflect_fix,18,0,0.444,0.200,NaN,0.111
3,critic,18,0,0.389,0.125,NaN,0.077


## 09 Cost per fix (`arm_outcomes.csv`)

In [19]:
df_09 = o[["arm", "cost_per_repair_usd", "tokens_per_repair", "extra_cost_per_extra_repair_usd"]]
df_09

,arm,cost_per_repair_usd,tokens_per_repair,extra_cost_per_extra_repair_usd
0,control,0.5592,52168,NaN
1,bare_retry,0.2867,28756,-0.2585
2,reflect_fix,0.2613,29914,-0.0367
3,critic,0.3019,38740,-0.0413


## 10 Accuracy vs cost (`curves.csv` + `arm_outcomes.csv` + `main.csv`)

In [20]:
allc = c[c["scope"] == "all"]
z = m[(m["system"] == "zero_shot") & (m["attempt"] == 0)].iloc[0]
pts = [("zero_shot", z["cost_usd"], allc[allc["attempt"] == 0]["hidden_pass@1"].iloc[0])]
oi = o.set_index("arm")
pts += [(a, oi.loc[a, "pipeline_cost_usd"],
         allc[(allc["arm"] == a) & (allc["attempt"] == LAST)]["hidden_pass@1"].iloc[0]) for a in ARMS]
pan = m[(m["system"] == "panacea") & (m["attempt"] == 0)]
if len(pan) and pan.iloc[0]["cost_usd"] > 0:
    pts.append(("panacea", pan.iloc[0]["cost_usd"], pan.iloc[0]["hidden_pass@1"]))
df_10 = pd.DataFrame(pts, columns=["system", "pipeline_cost_usd", "hidden_pass@1"])
df_10

,system,pipeline_cost_usd,hidden_pass@1
0,zero_shot,2.0849,0.77
1,control,4.3218,0.81
2,bare_retry,3.8049,0.83
3,reflect_fix,4.1752,0.85
4,critic,4.1980,0.84


## Other tables (no chart)
Every CSV is in `t`, e.g. `t["paired_tests"]`, `t["tasks"]`, `t["attempts"]`.

In [21]:
t["paired_tests"]

,family,comparison,attempt,n,first_passes,second_passes,only_first_passes,only_second_passes,both_pass,both_fail,p_exact_mcnemar,p_holm,significant
0,exploratory,bare_retry vs control,1,100,80,79,1,0,79,20,1.0000,1.0000,False
1,exploratory,reflect_fix vs control,1,100,83,79,5,1,78,16,0.2188,1.0000,False
2,exploratory,critic vs control,1,100,83,79,5,1,78,16,0.2188,1.0000,False
3,exploratory,bare_retry vs reflect_fix,1,100,80,83,1,4,79,16,0.3750,1.0000,False
4,exploratory,bare_retry vs critic,1,100,80,83,1,4,79,16,0.3750,1.0000,False
5,exploratory,reflect_fix vs critic,1,100,83,83,0,0,83,17,1.0000,1.0000,False
6,exploratory,bare_retry vs control,2,100,81,81,1,1,80,18,1.0000,1.0000,False
7,exploratory,reflect_fix vs control,2,100,85,81,5,1,80,14,0.2188,1.0000,False
8,exploratory,critic vs control,2,100,84,81,4,1,80,15,0.3750,1.0000,False
9,exploratory,bare_retry vs reflect_fix,2,100,81,85,1,5,80,14,0.2188,1.0000,False
